In [1]:
import pandas as pd
import geopandas as gpd
import time

MAP_KEY = '2790d08a201a95961b6d0d86c5131561'
sources = ['VIIRS_NOAA20_NRT', 'VIIRS_NOAA21_NRT']
AREA = '108.5,-4.5,119.5,7.5'

# Generate 5-day chunks from season start to today
date_starts = pd.date_range(start='2026-07-01', end='2026-09-17', freq='5D').strftime('%Y-%m-%d')

all_fires = []
for date in date_starts:
    for SOURCE in sources:
        url = f'https://firms.modaps.eosdis.nasa.gov/api/area/csv/{MAP_KEY}/{SOURCE}/{AREA}/5/{date}'
        chunk = pd.read_csv(url)
        all_fires.append(chunk)
        print(f"{date} ({SOURCE}): {len(chunk)} detections")
        time.sleep(1)

fire_df_season = pd.concat(all_fires, ignore_index=True).drop_duplicates()
print(f"\nTotal detections since July 1: {len(fire_df_season)}")

# Build GeoDataFrame
fire_gdf_season = gpd.GeoDataFrame(
    fire_df_season,
    geometry=gpd.points_from_xy(fire_df_season.longitude, fire_df_season.latitude),
    crs='EPSG:4326'
)

# Filter to reasonable confidence
fire_gdf_season = fire_gdf_season[fire_gdf_season['confidence'].isin(['n', 'h'])]

# Spatial join against confirmed-extant orangutan range (presence == 1)
fires_gdf_season = gpd.sjoin(fire_gdf_season, orangutan_range, predicate='within')
print(f"{len(fires_gdf_season)} detections within confirmed orangutan range since July 1")

# Buffer + dissolve to estimate affected area
fires_proj_season = fires_gdf_season.to_crs('EPSG:6933')
fires_proj_season['geometry'] = fires_proj_season.geometry.buffer(187.5)
burned_area_season = fires_proj_season.dissolve().geometry.area.sum() / 10_000

# Compare against total range
range_area_ha = orangutan_range.dissolve().to_crs('EPSG:6933').geometry.area.sum() / 10_000
pct_affected_season = burned_area_season / range_area_ha * 100

print(f"\nSince July 1, 2026:")
print(f"Approx. {burned_area_season:,.0f} hectares of confirmed orangutan range affected")
print(f"That's {pct_affected_season:.1f}% of total confirmed range ({range_area_ha:,.0f} ha)")

2026-07-01 (VIIRS_NOAA20_NRT): 429 detections
2026-07-01 (VIIRS_NOAA21_NRT): 411 detections
2026-07-06 (VIIRS_NOAA20_NRT): 1045 detections
2026-07-06 (VIIRS_NOAA21_NRT): 1003 detections
2026-07-11 (VIIRS_NOAA20_NRT): 1803 detections
2026-07-11 (VIIRS_NOAA21_NRT): 1870 detections
2026-07-16 (VIIRS_NOAA20_NRT): 1532 detections
2026-07-16 (VIIRS_NOAA21_NRT): 1582 detections
2026-07-21 (VIIRS_NOAA20_NRT): 2413 detections
2026-07-21 (VIIRS_NOAA21_NRT): 2493 detections
2026-07-26 (VIIRS_NOAA20_NRT): 1716 detections
2026-07-26 (VIIRS_NOAA21_NRT): 1641 detections
2026-07-31 (VIIRS_NOAA20_NRT): 5034 detections
2026-07-31 (VIIRS_NOAA21_NRT): 4824 detections
2026-08-05 (VIIRS_NOAA20_NRT): 11115 detections
2026-08-05 (VIIRS_NOAA21_NRT): 10363 detections
2026-08-10 (VIIRS_NOAA20_NRT): 9023 detections
2026-08-10 (VIIRS_NOAA21_NRT): 8746 detections
2026-08-15 (VIIRS_NOAA20_NRT): 16339 detections
2026-08-15 (VIIRS_NOAA21_NRT): 15869 detections
2026-08-20 (VIIRS_NOAA20_NRT): 13718 detections
2026-08-20

NameError: name 'orangutan_range' is not defined